In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model

import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
#  
data = pd.read_csv('MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv', header=0)

# 
X = data.iloc[:1000, :40]  # 
y = data.iloc[:1000, -1]   # 

# 
print("please check...")
print(X.isin([np.inf, -np.inf]).sum())  # 
print(X.isnull().sum())  # 

# 
X.replace([np.inf, -np.inf], np.nan, inplace=True)  # 
X.fillna(X.median(), inplace=True)  # 

# 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 
model = BayesianRidge()
model.fit(X_train_scaled, y_train)

# 
weights = model.coef_

# 
feature_weights = pd.DataFrame({
    'Feature': X.columns,  # 
    'Weight': weights
})

# 
feature_weights = feature_weights.sort_values(by='Weight', ascending=False)

# 
top_20_features = feature_weights.head(20)

# 
print("Top 20 features by weight:")
print(top_20_features)


import pandas as pd

# 
feature_weights = pd.DataFrame({
    'Feature': ['Total Backward Packets', 'Total Length of Bwd Packets'],
    'Weight': [0.000002, 0.000002]
})

# 
feature_weights.to_csv('MachineLearningCVE\Monday-feature_weights.csv', index=False, header=True)

print(" 'feature_weights.csv'")

In [ ]:
# # 
# print("Loading training data...")
# train_data = pd.read_csv("MachineLearningCVE\Monday-20.csv").head(1000)  # 
# print("Loading test data...")
# test_data = pd.read_csv("MachineLearningCVE/mormal.csv").head(1000)  # 

# 
print("Loading training data...")
train_data = pd.read_csv("MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv", usecols=range(40)).head(1000)  # 

print("Loading test data...")
test_data = pd.read_csv("MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv", usecols=range(40)).head(1000)  # 

# 
def handle_infinities_and_nans(df):
    # 
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # 
    df.fillna(df.mean(), inplace=True)
    return df

# 
print("Handling infinities and NaNs...")
train_data = handle_infinities_and_nans(train_data)
test_data = handle_infinities_and_nans(test_data)

# 
print("Standardizing data...")
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# 
train_scaled = np.array(train_scaled)
test_scaled = np.array(test_scaled)

# 
class TransformerAutoencoder(Model):
    def __init__(self, input_dim, num_heads=4, num_layers=2, dff=128, latent_dim=64):
        super(TransformerAutoencoder, self).__init__()
        self.input_dim = input_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dff = dff
        self.latent_dim = latent_dim
        
        # Encoder 
        self.encoder_input = layers.Input(shape=(None, input_dim))  # (None, input_dim)
        
        # Transformer 
        self.attn_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim) for _ in range(num_layers)]
        self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]
        
        # Decoder 
        self.decoder_output = layers.Dense(input_dim)
    
    def call(self, inputs):
        # Ensure the input is 3D tensor
        x = tf.expand_dims(inputs, axis=1)  # Add sequence length dimension (None, 1, input_dim)
        
        for i in range(self.num_layers):
            attn_output = self.attn_layers[i](x, x)  # 
            x = self.dense_layers[i](attn_output)  # 
        
        # Autoencoder
        reconstructed = self.decoder_output(x)
        return reconstructed


# 
print("Defining the model...")
input_dim = train_scaled.shape[1]  # 30
model = TransformerAutoencoder(input_dim=input_dim)

# 
print("Compiling and training the model...")
model.compile(optimizer=tf.keras.optimizers.Adam(), loss='mean_squared_error')

# 
model.fit(train_scaled, train_scaled, epochs=20, batch_size=64)

# 
def detect_anomalies(model, test_data, threshold=0.01):
    # 
    print("Detecting anomalies...")
    reconstructed = model.predict(test_data)
    
    # 
    mse = np.mean(np.square(reconstructed - test_data), axis=1)
    
    # 
    anomalies = np.where(mse > threshold)[0]  # 
    
    return anomalies, mse

# 
threshold = 0.01  # 
anomalies, mse = detect_anomalies(model, test_scaled, threshold=threshold)

# 
print(f"Detected anomalies at indices: {anomalies}")
print(f"Total number of anomalies detected: {len(anomalies)}")

In [ ]:
# #
# print("Loading training data...")
# train_data = pd.read_csv("MachineLearningCVE\Monday-20.csv").head(1000)  # 
# print("Loading test data...")
# test_data = pd.read_csv("MachineLearningCVE/mormal.csv").head(1000)  # 

# 
print("Loading training data...")
train_data = pd.read_csv("\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv", usecols=range(20)).head(1000)  # 

print("Loading test data...")
test_data = pd.read_csv("\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv", usecols=range(20)).head(1000)  # 

# 
def handle_infinities_and_nans(df):
    # 
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # 
    df.fillna(df.mean(), inplace=True)
    return df

# 
print("Handling infinities and NaNs...")
train_data = handle_infinities_and_nans(train_data)
test_data = handle_infinities_and_nans(test_data)

# 
print("Standardizing data...")
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# 
train_scaled = np.array(train_scaled)
test_scaled = np.array(test_scaled)

# 2
class TransformerAutoencoder(Model):
    def __init__(self, input_dim, num_heads=4, num_layers=2, dff=128, latent_dim=64):
        super(TransformerAutoencoder, self).__init__()
        self.input_dim = input_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dff = dff
        self.latent_dim = latent_dim
        
        #
        self.encoder_input = layers.Input(shape=(None, input_dim))  # (None, input_dim)
        
        # 
        self.attn_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim) for _ in range(num_layers)]
        self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]
        
        # Decoder 
        self.decoder_output = layers.Dense(input_dim)
    
    def call(self, inputs):
        # Ensure the input is 3D tensor
        x = tf.expand_dims(inputs, axis=1)  # Add sequence length dimension (None, 1, input_dim)
        
        for i in range(self.num_layers):
            attn_output = self.attn_layers[i](x, x)  # 
            x = self.dense_layers[i](attn_output)  # 
        
        # Autoencoder
        reconstructed = self.decoder_output(x)
        return reconstructed


# 3
print("Defining the model...")
input_dim = train_scaled.shape[1]  # 30
model = TransformerAutoencoder(input_dim=input_dim)

# 4
print("Compiling and training the model...")
model.compile(optimizer=tf.keras.optimizers.Adam(), loss='mean_squared_error')

# 
model.fit(train_scaled, train_scaled, epochs=20, batch_size=64)

# 5
def detect_anomalies(model, test_data, threshold=0.01):
    # 
    print("Detecting anomalies...")
    reconstructed = model.predict(test_data)
    
    # 
    mse = np.mean(np.square(reconstructed - test_data), axis=1)
    
    # 
    anomalies = np.where(mse > threshold)[0]  # 
    
    return anomalies, mse

# 6
threshold = 0.01  # 
anomalies, mse = detect_anomalies(model, test_scaled, threshold=threshold)

# 
print(f"Detected anomalies at indices: {anomalies}")
print(f"Total number of anomalies detected: {len(anomalies)}")

In [ ]:
def runTime():
    # 1
    print("Loading training data...")
    train_data = pd.read_csv("\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv", usecols=range(40)).head(5000)  # 
    print(f"Training data size: {train_data.shape[0]}")  # 
    print("Loading test data...")
    test_data = pd.read_csv("\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv", usecols=range(40)).head(5000)  # 
    # 
    def handle_infinities_and_nans(df):
        # 
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        # 
        df.fillna(df.mean(), inplace=True)
        return df

    # 
    print("Handling infinities and NaNs...")
    train_data = handle_infinities_and_nans(train_data)
    test_data = handle_infinities_and_nans(test_data)

    # 
    print("Standardizing data...")
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    test_scaled = scaler.transform(test_data)

    # 
    train_scaled = np.array(train_scaled)
    test_scaled = np.array(test_scaled)

    # 2
    class TransformerAutoencoder(Model):
        def __init__(self, input_dim, num_heads=4, num_layers=2, dff=128, latent_dim=64):
            super(TransformerAutoencoder, self).__init__()
            self.input_dim = input_dim
            self.num_heads = num_heads
            self.num_layers = num_layers
            self.dff = dff
            self.latent_dim = latent_dim
            
            # Encoder 
            self.encoder_input = layers.Input(shape=(None, input_dim))  #  (None, input_dim)
            
            # Transformer 
            self.attn_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim) for _ in range(num_layers)]
            self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]
            
            # Decoder 
            self.decoder_output = layers.Dense(input_dim)
        
        def call(self, inputs):
            # Ensure the input is 3D tensor
            x = tf.expand_dims(inputs, axis=1)  # Add sequence length dimension (None, 1, input_dim)
            
            for i in range(self.num_layers):
                attn_output = self.attn_layers[i](x, x)  # 
                x = self.dense_layers[i](attn_output)  # 
            
            # Autoencoder
            reconstructed = self.decoder_output(x)
            return reconstructed


    # 3
    print("Defining the model...")
    input_dim = train_scaled.shape[1]  # 30
    model = TransformerAutoencoder(input_dim=input_dim)

    # 4
    print("Compiling and training the model...")
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss='mean_squared_error')

    # 
    model.fit(train_scaled, train_scaled, epochs=20, batch_size=64)

    # 5
    def detect_anomalies(model, test_data, threshold=0.01):
        # 
        print("Detecting anomalies...")
        reconstructed = model.predict(test_data)
        
        # 
        mse = np.mean(np.square(reconstructed - test_data), axis=1)
        
        # 
        anomalies = np.where(mse > threshold)[0]  # 
        
        return anomalies, mse

    # 6
    threshold = 0.01  # 
    anomalies, mse = detect_anomalies(model, test_scaled, threshold=threshold)

    # 
    print(f"Detected anomalies at indices: {anomalies}")
    print(f"Total number of anomalies detected: {len(anomalies)}")

import timeit


# 
execution_time = timeit.timeit(runTime, number=1)

print(f" {execution_time} ")

In [ ]:
def runTime():
    # 1
    print("Loading training data...")
    train_data = pd.read_csv("\MachineLearningCVE\Monday-WorkingHours.pcap_ISCX.csv", usecols=range(20)).head(5000)  # 
    print(f"Training data size: {train_data.shape[0]}")  # 
    print("Loading test data...")
    test_data = pd.read_csv("\MachineLearningCVE\Tuesday-WorkingHours.pcap_ISCX.csv", usecols=range(20)).head(5000)  # 
    print(f"Training data size: {test_data.shape[0]}")  # 
    # 
    def handle_infinities_and_nans(df):
        # 
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        # 
        df.fillna(df.mean(), inplace=True)
        return df

    # 
    # print("Handling infinities and NaNs...")
    train_data = handle_infinities_and_nans(train_data)
    test_data = handle_infinities_and_nans(test_data)

    # 
    # print("Standardizing data...")
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    test_scaled = scaler.transform(test_data)

    # 
    train_scaled = np.array(train_scaled)
    test_scaled = np.array(test_scaled)

    # 2
    class TransformerAutoencoder(Model):
        def __init__(self, input_dim, num_heads=4, num_layers=2, dff=128, latent_dim=64):
            super(TransformerAutoencoder, self).__init__()
            self.input_dim = input_dim
            self.num_heads = num_heads
            self.num_layers = num_layers
            self.dff = dff
            self.latent_dim = latent_dim
            
            # Encoder 
            self.encoder_input = layers.Input(shape=(None, input_dim))  # (None, input_dim)
            
            # Transformer 
            self.attn_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim) for _ in range(num_layers)]
            self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]
            
            # Decoder 
            self.decoder_output = layers.Dense(input_dim)
        
        def call(self, inputs):
            # Ensure the input is 3D tensor
            x = tf.expand_dims(inputs, axis=1)  # Add sequence length dimension (None, 1, input_dim)
            
            for i in range(self.num_layers):
                attn_output = self.attn_layers[i](x, x)  # 
                x = self.dense_layers[i](attn_output)  # 
            
            # Autoencoder
            reconstructed = self.decoder_output(x)
            return reconstructed


    # 
    # print("Defining the model...")
    input_dim = train_scaled.shape[1]  # 30
    model = TransformerAutoencoder(input_dim=input_dim)

    # 4
    # print("Compiling and training the model...")
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss='mean_squared_error')

    # 
    model.fit(train_scaled, train_scaled, epochs=20, batch_size=64)

    # 5
    def detect_anomalies(model, test_data, threshold=0.01):
        # 
        # print("Detecting anomalies...")
        reconstructed = model.predict(test_data)
        
        # 
        mse = np.mean(np.square(reconstructed - test_data), axis=1)
        
        # 
        anomalies = np.where(mse > threshold)[0]  # 
        
        return anomalies, mse

    # 6
    threshold = 0.01  # 
    anomalies, mse = detect_anomalies(model, test_scaled, threshold=threshold)

    # 
    # print(f"Detected anomalies at indices: {anomalies}")
    # print(f"Total number of anomalies detected: {len(anomalies)}")

import timeit


# 
execution_time = timeit.timeit(runTime, number=1)

print(f"{execution_time} ")

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
import timeit

# 
def load_data(file_path, chunk_size=5000, usecols=None, nrows=None):
    print(f"Loading data from {file_path} in chunks of size {chunk_size}...")
    for chunk in pd.read_csv(file_path, usecols=usecols, chunksize=chunk_size, nrows=nrows):
        # 
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.fillna(chunk.mean(), inplace=True)
        yield chunk

def runTime(nrows=None, num_columns=None):
    # 1
    print("Loading training data...")
    # 
    train_data_chunks = load_data("/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv", 
                                  usecols=range(num_columns), nrows=nrows)
    train_data = pd.concat(train_data_chunks)
    print(f"Training data size: {train_data.shape[0]}")  # 

    print("Loading test data...")
    test_data_chunks = load_data("/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv", 
                                 usecols=range(num_columns), nrows=nrows)
    test_data = pd.concat(test_data_chunks)
    print(f"Test data size: {test_data.shape[0]}")  # 

    # 
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_data)
    test_scaled = scaler.transform(test_data)

    # 2
    class TransformerAutoencoder(Model):
        def __init__(self, input_dim, num_heads=4, num_layers=2, dff=128, latent_dim=64):
            super(TransformerAutoencoder, self).__init__()
            self.input_dim = input_dim
            self.num_heads = num_heads
            self.num_layers = num_layers
            self.dff = dff
            self.latent_dim = latent_dim

            # Encoder
            self.encoder_input = layers.Input(shape=(None, input_dim))  # (None, input_dim)

            # Transformer 
            self.attn_layers = [layers.MultiHeadAttention(num_heads=num_heads, key_dim=latent_dim) for _ in range(num_layers)]
            self.dense_layers = [layers.Dense(dff, activation='relu') for _ in range(num_layers)]

            # Decoder 
            self.decoder_output = layers.Dense(input_dim)

        def call(self, inputs):
            # Ensure the input is 3D tensor
            x = tf.expand_dims(inputs, axis=1)  # Add sequence length dimension (None, 1, input_dim)

            for i in range(self.num_layers):
                attn_output = self.attn_layers[i](x, x)  # 
                x = self.dense_layers[i](attn_output)  # 

            # Autoencoder
            reconstructed = self.decoder_output(x)
            return reconstructed

    # 3
    input_dim = train_scaled.shape[1]  # num_columns
    model = TransformerAutoencoder(input_dim=input_dim)

    # 4
    model.compile(optimizer=tf.keras.optimizers.Adam(), loss='mean_squared_error')

    # 
    epochs = 20
    batch_size = 64

    # 
    model.fit(train_scaled, train_scaled, epochs=epochs, batch_size=batch_size, validation_data=(test_scaled, test_scaled))
    
    print(f" {execution_time} ")
    # 
    def detect_anomalies(model, test_data, threshold=0.01):
        # 
        reconstructed = model.predict(test_data)

        # 
        mse = np.mean(np.square(reconstructed - test_data), axis=1)

        # 
        anomalies = np.where(mse > threshold)[0]  # 

        return anomalies, mse
    print(f" {execution_time} ")

    # 
    threshold = 0.01  # 
    anomalies, mse = detect_anomalies(model, test_scaled, threshold=threshold)

    # 
    print(f"Detected anomalies at indices: {anomalies}")
    print(f"Total number of anomalies detected: {len(anomalies)}")

# 
execution_time = timeit.timeit(lambda: runTime(nrows=5000, num_columns=20), number=1)

print(f"time: {execution_time} s")


In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# 
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (5,3)

# 
X = [1000, 2000, 3000, 4000, 5000]
Y1 = [5.63145330000043, 7.13297389999934, 12.1976284000011, 17.1976284000011, 22.535687646]
Y2 = [5.10881870000048, 6.52167920000101, 9.78975150000042, 12.1573853, 14.355245768]

# 
x = np.arange(len(X))
width = 0.35

# 
y1_x = x
y2_x = x + width

# 
plt.bar(y1_x, Y1, width=width, color="#A6C8FF", label="No Bayesian")  # 
plt.bar(y2_x, Y2, width=width, color="#FFDBA8", label="Bayesian")  # 

# 
plt.xticks(x + width / 2, labels=[f'{x}' for x in X])

# 
for i in range(len(X)):
    plt.text(y1_x[i], Y1[i], f'{Y1[i]:.2f}', va="bottom", ha="center", fontsize=8)
    plt.text(y2_x[i], Y2[i], f'{Y2[i]:.2f}', va="bottom", ha="center", fontsize=8)

# 
plt.legend(loc="upper left")

# 
plt.tight_layout()  # 
plt.show()
